# 02d — Microbiology features

Calculate culture timing and ESBL status around admission. Duplicate spells are reduced before assigning episode-level features.

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/app/notebooks/data/synthetic/10000p/")
KEYS = ["SUBJECT", "SPELL_IDENTIFIER"]

def keep_first_spell(df: pd.DataFrame, name: str = "episodes") -> pd.DataFrame:
    """Keep the first source row for each SUBJECT/SPELL_IDENTIFIER pair."""
    missing = [c for c in KEYS if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing key columns: {missing}")
    result = df.copy()
    before = len(result)
    result = result.drop_duplicates(KEYS, keep="first").reset_index(drop=True)
    print(f"{name}: {before:,} rows -> {len(result):,} unique spells")
    return result

print(f"Using data from: {DATA_DIR}")


Using data from: /app/notebooks/data/synthetic/10000p


In [13]:
from icare_risk.clinphen.utils import (
    compute_time_to_closest_culture,
    assign_esbl_to_episodes,
)

df_episodes = pd.read_csv(
    DATA_DIR / "icare_episodes_anon.csv",
    parse_dates=["ADMISSION_DATE", "DISCHARGE_DATE"],
)
df_micro = pd.read_csv(DATA_DIR / "icare_microbiology_anon.csv")
df_episodes = keep_first_spell(df_episodes)


episodes: 10,000 rows -> 10,000 unique spells


In [9]:
df_micro_timing = compute_time_to_closest_culture(
    df_episodes=df_episodes,
    microbiology_df=df_micro,
    pre_admission_days=14,
    post_discharge_days=7,
    round_decimals=0,
)

LOOKUP_3GCR = Path("../../data/lookups/standard/3GCR.csv")
LOOKUP_3GCS = Path("../../data/lookups/standard/3GCS.csv")

df_microbiology = assign_esbl_to_episodes(
    df_episodes=df_micro_timing,
    df_microbiology=df_micro,
    path_3gcr=str(LOOKUP_3GCR),
    path_3gcs=str(LOOKUP_3GCS),
    pre_admission_days=14,
    post_discharge_days=7,
)
df_microbiology = keep_first_spell(df_microbiology, "microbiology features")
display(df_microbiology.head())


Computing test-level ESBL labels on the fly...
microbiology features: 10,000 rows -> 10,000 unique spells


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,closest_culture_date,time_to_culture_days,time_to_culture_hours,esbl_status
0,10001,7929089,2005940,4725136,2020-06-10,22:22:00,35,2020-07-03,320,cardiology,7,NaN,NaN,NaN,unknown
1,10002,8200425,6570033,6896867,2020-03-27,17:31:00,63,2020-03-28,301,gastroenterology,6,NaN,NaN,NaN,unknown
2,10003,9931252,5492977,1426665,2023-01-06,00:18:00,89,2023-01-25,301,gastroenterology,5,NaN,NaN,NaN,unknown
3,10004,5095152,3730006,3105590,2022-09-14,10:32:00,25,2022-10-12,301,gastroenterology,4,NaN,NaN,NaN,unknown
4,10005,7807609,7413177,1104787,2023-09-08,10:55:00,80,2023-09-23,140,oral surgery,none,NaN,NaN,NaN,unknown


In [14]:
output_path = DATA_DIR / "02e_derived_features_microbiology.parquet"
df_microbiology.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /app/notebooks/data/synthetic/10000p/02d_derived_features_microbiology.parquet
